# T2 BLIP Base — full Flickr30k pseudo-caption generation

Full run được phát hành sau pilot đạt gate. Ba notebook T1/T2/T3 độc lập và có
thể chạy song song. T1/T2 chỉ dùng Validation để báo cáo zero-shot transfer,
không chọn checkpoint; T3 dùng Validation CIDEr-D để chọn decoder. Held-out
Test luôn niêm phong.

## Kaggle input bắt buộc — đúng 8 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_T2_Handoff`.
4. Output `phase3_teacher_b_seed42`.
5. Dataset `Flickr Image dataset`.
6. Dataset/Kaggle Model BLIP Base offline có `MODEL_SOURCE_MANIFEST.json`.
7. Output `F30K_T2_Pseudo_Pilot`.
8. Package/dataset `pycocoevalcap-1.2` offline.

Khi resume, gắn thêm đúng một output gần nhất của chính notebook này.

## Runtime

- Accelerator: GPU T4; một GPU là đủ.
- Internet: Off.
- Khoảng 1,2–2,0 giờ theo pilot; cộng thời gian khởi tạo/ghi artifact.


## Cell 1 — Full-run authorization và Flickr30k contract

In [1]:

import os
EXPERIMENT_ID = "F30K_T2_Pseudo_Full"
TEACHER_ID = "T2"
HANDOFF_RESULT_NAME = "F30K_T2_Handoff_RESULT.json"
PILOT_RESULT_NAME = "F30K_T2_Pseudo_Pilot_RESULT.json"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import csv, hashlib, json, math, os, random, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def image_index(required_names):
    candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in required_names]
    by_name = {}
    duplicate_aliases = 0
    for path in candidates:
        old = by_name.get(path.name)
        if old is None:
            by_name[path.name] = path
            continue
        if old.resolve() == path.resolve():
            continue
        assert old.stat().st_size == path.stat().st_size, (
            "Ảnh trùng tên nhưng khác nội dung/kích thước", path.name, old, path
        )
        duplicate_aliases += 1
        by_name[path.name] = min(
            (old, path), key=lambda p: (len(p.parts), len(str(p)), str(p))
        )
    missing = sorted(required_names - set(by_name))
    assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])
    return by_name, duplicate_aliases

DATA_PATH = unique_named("f30k_train.parquet")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
HANDOFF_PATH = unique_named(HANDOFF_RESULT_NAME)

data_contract = read_json(DATA_CONTRACT_PATH)
base_ready = read_json(BASE_READY_PATH)
vocab = read_json(VOCAB_PATH)
handoff = read_json(HANDOFF_PATH)

assert data_contract["status"] == "complete"
assert data_contract["train_image_count"] == 29000
assert data_contract["validation_image_count"] == 1014
assert data_contract["test_image_count"] == 1000
assert data_contract["vocabulary_size"] == 7456
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False
assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert handoff["teacher_id"] == TEACHER_ID
assert handoff["flickr30k"]["test_set_status"] == "sealed"
assert handoff["flickr30k"]["test_set_used"] is False
assert sha256(DATA_PATH) == data_contract["artifacts"]["train"]["sha256"]
assert sha256(VOCAB_PATH) == data_contract["artifacts"]["vocabulary"]["sha256"]

train_df = pd.read_parquet(DATA_PATH)
assert train_df["image_id"].nunique() == 29000
assert len(train_df) == 145000
train_ids = sorted(int(x) for x in train_df["image_id"].unique())
required_names = {f"{image_id}.jpg" for image_id in train_ids}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(required_names)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), "Notebook pilot yêu cầu GPU T4."
DEVICE = torch.device("cuda:0")

print("Common gate             : PASSED")
print("Experiment              :", EXPERIMENT_ID)
print("Teacher                 :", TEACHER_ID)
print("Flickr30k Train         : 29,000 images")
print("Vocabulary              : 7,456")
print("Nested image aliases    :", DUPLICATE_ALIASES, "(deduplicated)")
print("Held-out Test           : SEALED / NOT READ")


Common gate             : PASSED
Experiment              : F30K_T2_Pseudo_Full
Teacher                 : T2
Flickr30k Train         : 29,000 images
Vocabulary              : 7,456
Nested image aliases    : 29000 (deduplicated)
Held-out Test           : SEALED / NOT READ


## Cell 2 — Production workload

In [2]:

from torch.utils.data import DataLoader, Dataset
from transformers import BlipForConditionalGeneration, BlipProcessor

assert handoff["status"] == "passed"
assert handoff["method_contract"]["selected_method"] == "Sequence KD"
EXPECTED_CHECKPOINT_SHA = handoff["teacher_source"]["development_checkpoint_sha256"]
CHECKPOINT_PATH = unique_named("teacher_b_seed42_best.pt")
assert sha256(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA

model_sources = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(manifest_path)
    except Exception:
        continue
    if (
        payload.get("repo_id") == "Salesforce/blip-image-captioning-base"
        and payload.get("resolved_commit") == "82a37760796d32b1411fe092ab5d4e227313294b"
    ):
        model_sources.append(manifest_path.parent)
assert len(model_sources) == 1, model_sources
MODEL_SOURCE = model_sources[0]

processor = BlipProcessor.from_pretrained(
    str(MODEL_SOURCE), local_files_only=True, use_fast=False
)
tokenizer = processor.tokenizer
if "[DEC]" not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"bos_token": "[DEC]"})
else:
    tokenizer.bos_token = "[DEC]"
if "[ENC]" not in tokenizer.get_vocab():
    current = tokenizer.special_tokens_map.get("additional_special_tokens", [])
    if isinstance(current, str): current = [current]
    tokenizer.add_special_tokens({"additional_special_tokens": list(current) + ["[ENC]"]})
assert tokenizer.convert_tokens_to_ids("[DEC]") == 30522
assert tokenizer.convert_tokens_to_ids("[ENC]") == 30523

model = BlipForConditionalGeneration.from_pretrained(
    str(MODEL_SOURCE), local_files_only=True
)
payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
assert payload["schema_version"] == "teacher_b_seed42_best_model_v1"
assert payload["seed"] == 42 and payload["test_set_used"] is False
load_result = model.load_state_dict(payload["model"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
model = model.to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)


def load_image(image_id):
    with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
        return image.convert("RGB").copy()
def full_collate(rows):
    indices, ids, images = zip(*rows)
    values = processor(images=list(images), return_tensors="pt")["pixel_values"]
    return torch.tensor(indices), torch.tensor(ids), values
def generate_batch(images):
    images = images.to(DEVICE, non_blocking=True)
    with torch.inference_mode():
        sequences = model.generate(
            pixel_values=images, num_beams=3, do_sample=False,
            length_penalty=0.7, max_new_tokens=31, min_new_tokens=2,
            early_stopping=True,
        )
    ids = [[int(x) for x in row] for row in sequences.detach().cpu().tolist()]
    captions = [x.strip() for x in processor.batch_decode(sequences, skip_special_tokens=True)]
    return ids, captions
CHECKPOINT_SHA = sha256(CHECKPOINT_PATH)
BATCH_SIZE = 8
CHECKPOINT_EVERY_BATCHES = 250
VALIDATION_NUM_WORKERS = 0
VALIDATION_COLLATE = full_collate
TEACHER_MODEL = "BLIP Base"
MATCHED_STUDENT = "S2 MobileViT-S"
NEXT_PHASE = "F30K_T2S2_SeqKD_42"


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

## Cell 3 — Production workload

In [3]:

import importlib.util, re, shutil, subprocess
import pyarrow as pa
import pyarrow.parquet as pq
from torch.utils.data import DataLoader, Dataset

PILOT_RESULT_PATH = unique_named(PILOT_RESULT_NAME)
pilot_result = read_json(PILOT_RESULT_PATH)
assert pilot_result["status"] == "passed"
assert pilot_result["pilot_gate"] == "full_pseudocaption_generation_authorized"
assert pilot_result["next_phase_authorized"] == EXPERIMENT_ID
assert pilot_result["test_set_status"] == "sealed"
assert pilot_result["test_set_used"] is False

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?|[^\w\s]", re.UNICODE)
TOKEN_TO_ID = vocab["token_to_id"]
def to_f30k_ids(caption):
    lexical = [x.lower() for x in TOKEN_RE.findall(caption.strip())][:29]
    return [1] + [int(TOKEN_TO_ID.get(x, 3)) for x in lexical] + [2]

VAL_PATH = unique_named("f30k_validation.parquet")
assert sha256(VAL_PATH) == data_contract["artifacts"]["validation"]["sha256"]
val_df = pd.read_parquet(VAL_PATH)
assert val_df["image_id"].nunique() == 1014 and len(val_df) == 5070
VAL_REFERENCES = {
    int(image_id): [str(x) for x in group.caption.tolist()]
    for image_id, group in val_df.groupby("image_id", sort=True)
}
all_names = required_names | {f"{image_id}.jpg" for image_id in VAL_REFERENCES}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(all_names)

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))
assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR yêu cầu Java runtime."

class FullImages(Dataset):
    def __init__(self, start_index):
        self.start_index = int(start_index)
    def __len__(self):
        return len(train_ids) - self.start_index
    def __getitem__(self, relative_index):
        global_index = self.start_index + relative_index
        image_id = train_ids[global_index]
        return global_index, image_id, load_image(image_id)

PARTIAL_PATH = WORK_ROOT / "pseudocaptions.partial.jsonl"
FINAL_JSONL = WORK_ROOT / "pseudocaptions.jsonl"
PROGRESS_PATH = WORK_ROOT / "PROGRESS.json"
progress_candidates = []
for candidate in INPUT_ROOT.rglob("PROGRESS.json"):
    try:
        payload = read_json(candidate)
    except Exception:
        continue
    if (
        payload.get("experiment_id") == EXPERIMENT_ID
        and payload.get("checkpoint_sha256") == CHECKPOINT_SHA
        and payload.get("total_records") == 29000
        and payload.get("status") == "in_progress"
    ):
        progress_candidates.append((candidate, payload))

completed = 0
resume_used = False
if progress_candidates:
    progress_candidates.sort(key=lambda item: int(item[1]["completed_records"]))
    source_progress, previous = progress_candidates[-1]
    source_partial = source_progress.parent / previous["partial_relative_path"]
    assert source_partial.is_file()
    assert source_partial.stat().st_size == int(previous["partial_size_bytes"])
    assert sha256(source_partial) == previous["partial_sha256"]
    shutil.copy2(source_partial, PARTIAL_PATH)
    with PARTIAL_PATH.open("r", encoding="utf-8") as handle:
        for expected, line in enumerate(handle):
            row = json.loads(line)
            assert row["global_index"] == expected
            assert row["image_id"] == train_ids[expected]
            completed = expected + 1
    assert completed == int(previous["completed_records"])
    resume_used = True

loader = DataLoader(
    FullImages(completed), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=full_collate,
)
mode = "a" if resume_used else "w"
started = time.perf_counter()
starting_record = completed
with PARTIAL_PATH.open(mode, encoding="utf-8", newline="\n") as handle:
    for batch_index, batch in enumerate(loader, start=1):
        global_indices, image_ids, images = batch
        teacher_ids, captions = generate_batch(images)
        for global_index, image_id, native_ids, caption in zip(
            global_indices.tolist(), image_ids.tolist(), teacher_ids, captions
        ):
            assert int(global_index) == completed
            assert int(image_id) == train_ids[completed]
            student_ids = to_f30k_ids(caption)
            assert caption and student_ids[0] == 1 and student_ids[-1] == 2
            assert len(student_ids) <= 31 and 0 not in student_ids
            handle.write(json.dumps({
                "global_index": completed,
                "image_id": int(image_id),
                "caption": caption,
                "teacher_token_ids": [int(x) for x in native_ids],
                "student_token_ids": student_ids,
                "teacher_checkpoint_sha256": CHECKPOINT_SHA,
            }, ensure_ascii=False) + "\n")
            completed += 1
        if batch_index % CHECKPOINT_EVERY_BATCHES == 0 or completed == 29000:
            handle.flush()
            os.fsync(handle.fileno())
            elapsed = time.perf_counter() - started
            rate = (completed - starting_record) / max(elapsed, 1e-9)
            write_json(PROGRESS_PATH, {
                "schema_version": "f30k_pseudocaption_progress_v1",
                "created_at_utc": datetime.now(timezone.utc).isoformat(),
                "experiment_id": EXPERIMENT_ID, "status": "in_progress",
                "teacher_id": TEACHER_ID, "checkpoint_sha256": CHECKPOINT_SHA,
                "total_records": 29000, "completed_records": completed,
                "partial_relative_path": PARTIAL_PATH.name,
                "partial_size_bytes": PARTIAL_PATH.stat().st_size,
                "partial_sha256": sha256(PARTIAL_PATH),
                "images_per_second_this_run": rate,
                "estimated_remaining_hours": (29000 - completed) / max(rate, 1e-9) / 3600,
                "test_set_status": "sealed", "test_set_used": False,
            })
            print(
                f"{completed:05d}/29000 | {rate:.3f} image/s | "
                f"ETA={(29000-completed)/max(rate,1e-9)/3600:.2f}h",
                flush=True,
            )

assert completed == 29000
os.replace(PARTIAL_PATH, FINAL_JSONL)
write_json(PROGRESS_PATH, {
    "schema_version": "f30k_pseudocaption_progress_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": EXPERIMENT_ID, "status": "generation_complete",
    "teacher_id": TEACHER_ID, "checkpoint_sha256": CHECKPOINT_SHA,
    "total_records": 29000, "completed_records": 29000,
    "final_relative_path": FINAL_JSONL.name,
    "final_size_bytes": FINAL_JSONL.stat().st_size,
    "final_sha256": sha256(FINAL_JSONL),
    "test_set_status": "sealed", "test_set_used": False,
})


02000/29000 | 5.623 image/s | ETA=1.33h
04000/29000 | 5.789 image/s | ETA=1.20h
06000/29000 | 5.855 image/s | ETA=1.09h
08000/29000 | 5.883 image/s | ETA=0.99h
10000/29000 | 5.917 image/s | ETA=0.89h
12000/29000 | 5.926 image/s | ETA=0.80h
14000/29000 | 5.948 image/s | ETA=0.70h
16000/29000 | 5.959 image/s | ETA=0.61h
18000/29000 | 5.969 image/s | ETA=0.51h
20000/29000 | 5.972 image/s | ETA=0.42h
22000/29000 | 5.959 image/s | ETA=0.33h
24000/29000 | 5.948 image/s | ETA=0.23h
26000/29000 | 5.938 image/s | ETA=0.14h
28000/29000 | 5.941 image/s | ETA=0.05h
29000/29000 | 5.946 image/s | ETA=0.00h


## Cell 4 — Production workload

In [4]:

class ValidationImages(Dataset):
    def __init__(self):
        self.ids = sorted(VAL_REFERENCES)
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        image_id = self.ids[index]
        return index, image_id, load_image(image_id)

validation_loader = DataLoader(
    ValidationImages(), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=VALIDATION_NUM_WORKERS, pin_memory=True,
    collate_fn=VALIDATION_COLLATE,
)
validation_hypotheses = {}
for _, image_ids, images in validation_loader:
    _, captions = generate_batch(images)
    for image_id, caption in zip(image_ids.tolist(), captions):
        assert caption
        validation_hypotheses[int(image_id)] = [caption]
assert len(validation_hypotheses) == 1014

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
references = {
    key: [{"caption": x} for x in values]
    for key, values in VAL_REFERENCES.items()
}
hypotheses = {
    key: [{"caption": values[0]}]
    for key, values in validation_hypotheses.items()
}
tokenizer = PTBTokenizer()
references = tokenizer.tokenize(references)
hypotheses = tokenizer.tokenize(hypotheses)
bleu, _ = Bleu(4).compute_score(references, hypotheses)
meteor = Meteor()
try:
    meteor_value, _ = meteor.compute_score(references, hypotheses)
finally:
    close = getattr(meteor, "close", None)
    if callable(close): close()
rouge, _ = Rouge().compute_score(references, hypotheses)
cider, _ = Cider().compute_score(references, hypotheses)
TEACHER_VALIDATION_METRICS = {
    "BLEU_1": bleu[0], "BLEU_4": bleu[3], "METEOR": meteor_value,
    "ROUGE_L": rouge, "CIDEr_D": cider,
}
(WORK_ROOT / "validation_captions.json").write_text(
    json.dumps({
        "references": VAL_REFERENCES,
        "hypotheses": validation_hypotheses,
    }, ensure_ascii=False),
    encoding="utf-8",
)
print("Validation images       : 1,014")
print("Transfer CIDEr-D        :", TEACHER_VALIDATION_METRICS["CIDEr_D"])


PTBTokenizer tokenized 74030 tokens at 311442.10 tokens per second.
PTBTokenizer tokenized 12030 tokens at 115949.03 tokens per second.


{'testlen': 10050, 'reflen': 10029, 'guess': [10050, 9036, 8022, 7008], 'correct': [7268, 3613, 1614, 715]}
ratio: 1.0020939276098313
Validation images       : 1,014
Transfer CIDEr-D        : 0.7017102633524122


## Cell 5 — Production workload

In [5]:

records = []
unknown_tokens = 0
lexical_tokens = 0
with FINAL_JSONL.open("r", encoding="utf-8") as handle:
    for expected, line in enumerate(handle):
        row = json.loads(line)
        assert row["global_index"] == expected
        assert row["image_id"] == train_ids[expected]
        assert row["teacher_checkpoint_sha256"] == CHECKPOINT_SHA
        ids = [int(x) for x in row["student_token_ids"]]
        assert ids[0] == 1 and ids[-1] == 2 and len(ids) <= 31 and 0 not in ids
        unknown_tokens += sum(x == 3 for x in ids[1:-1])
        lexical_tokens += len(ids) - 2
        records.append(row)
assert len(records) == 29000
assert len({row["image_id"] for row in records}) == 29000

PARQUET_PATH = WORK_ROOT / "pseudocaptions.parquet"
pq.write_table(
    pa.Table.from_pydict({
        "global_index": [x["global_index"] for x in records],
        "image_id": [x["image_id"] for x in records],
        "caption": [x["caption"] for x in records],
        "teacher_token_ids": [x["teacher_token_ids"] for x in records],
        "student_token_ids": [x["student_token_ids"] for x in records],
        "teacher_checkpoint_sha256": [x["teacher_checkpoint_sha256"] for x in records],
    }),
    PARQUET_PATH, compression="zstd", row_group_size=4096,
)
assert pq.read_table(PARQUET_PATH).num_rows == 29000

RESULT = {
    "schema_version": "f30k_pseudocaption_full_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete", "experiment_id": EXPERIMENT_ID,
    "teacher_id": TEACHER_ID, "teacher_model": TEACHER_MODEL,
    "matched_student": MATCHED_STUDENT, "method": "Sequence KD",
    "records": 29000, "unique_image_ids": 29000,
    "checkpoint_sha256": CHECKPOINT_SHA,
    "beam_size": 3, "length_penalty": 0.7,
    "maximum_decoder_steps": 31,
    "unknown_token_rate": unknown_tokens / max(1, lexical_tokens),
    "validation_images": 1014,
    "validation_metrics": TEACHER_VALIDATION_METRICS,
    "validation_role": "zero-shot transfer baseline; not checkpoint selection",
    "jsonl": {"name": FINAL_JSONL.name, "bytes": FINAL_JSONL.stat().st_size, "sha256": sha256(FINAL_JSONL)},
    "parquet": {"name": PARQUET_PATH.name, "bytes": PARQUET_PATH.stat().st_size, "sha256": sha256(PARQUET_PATH)},
    "sequence_kd_ready": True,
    "next_phase_authorized": NEXT_PHASE,
    "test_set_status": "sealed", "test_set_used": False,
}
write_json(WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json", RESULT)

artifact_paths = sorted(
    p for p in WORK_ROOT.rglob("*")
    if p.is_file() and p.name not in {"MANIFEST.json"}
)
manifest = {
    "schema_version": "f30k_pseudocaption_full_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": EXPERIMENT_ID, "status": "complete",
    "artifacts": [
        {"relative_path": p.relative_to(WORK_ROOT).as_posix(),
         "size_bytes": p.stat().st_size, "sha256": sha256(p)}
        for p in artifact_paths
    ],
    "test_set_status": "sealed", "test_set_used": False,
}
write_json(WORK_ROOT / "MANIFEST.json", manifest)
print("=" * 78)
print(EXPERIMENT_ID, "— FINAL")
print("=" * 78)
print("Status                  : COMPLETE")
print("Pseudo-caption records  : 29,000 / 29,000")
print("JSONL / Parquet         : READY / READY")
print("Validation CIDEr-D      :", TEACHER_VALIDATION_METRICS["CIDEr_D"])
print("Sequence KD             : READY")
print("Next                    :", NEXT_PHASE)
print("Held-out Test           : SEALED / NOT READ")
print("Output                  :", WORK_ROOT)


F30K_T2_Pseudo_Full — FINAL
Status                  : COMPLETE
Pseudo-caption records  : 29,000 / 29,000
JSONL / Parquet         : READY / READY
Validation CIDEr-D      : 0.7017102633524122
Sequence KD             : READY
Next                    : F30K_T2S2_SeqKD_42
Held-out Test           : SEALED / NOT READ
Output                  : /kaggle/working/F30K_T2_Pseudo_Full
